In [0]:
exec(open("/Workspace/ecommerce_retail/config.py").read())

df_payments = spark.read \
    .option("fs.s3a.access.key", ACCESS_KEY) \
    .option("fs.s3a.secret.key", SECRET_KEY) \
    .option("fs.s3a.session.token", SESSION_TOKEN) \
    .option("fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.TemporaryAWSCredentialsProvider") \
    .parquet(f"{S3_GOLD_BASE}/payment_summary/")

df_payments.createOrReplaceTempView("payment_summary")
print(f"✅ {df_payments.count()} rows loaded")
df_payments.printSchema()

✅ Credentials loaded (use as .option() in read/write calls)
✅ 5895 rows loaded
root
 |-- payment_date: date (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- transaction_count: long (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- avg_transaction_value: double (nullable = true)
 |-- avg_fraud_score: double (nullable = true)
 |-- etl_timestamp: timestamp (nullable = true)
 |-- payment_method: string (nullable = true)



In [0]:
%pip install plotly pandas
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
df = spark.sql("""
    SELECT
        COUNT(*)                                                   AS total_transactions,
        ROUND(SUM(total_amount), 2)                                AS total_amount_processed,
        ROUND(AVG(total_amount), 2)                                AS avg_transaction_value,
        ROUND(AVG(avg_fraud_score), 4)                             AS avg_fraud_score,
        COUNT(DISTINCT payment_method)                             AS payment_methods_count,
        COUNT(DISTINCT payment_status)                             AS status_count,
        COUNT(CASE WHEN payment_status = 'completed' THEN 1 END)   AS completed_transactions,
        COUNT(CASE WHEN payment_status = 'failed'    THEN 1 END)   AS failed_transactions,
        COUNT(CASE WHEN payment_status = 'refunded'  THEN 1 END)   AS refunded_transactions,
        COUNT(CASE WHEN payment_status = 'pending'   THEN 1 END)   AS pending_transactions
    FROM payment_summary
""").toPandas()

kpi = df.T.reset_index()
kpi.columns = ["KPI", "Value"]

fig = go.Figure(data=[go.Table(
    header=dict(
        values=["<b>KPI</b>", "<b>Value</b>"],
        fill_color="#2E3A59",
        font=dict(color="white", size=13),
        align="left",
        height=35
    ),
    cells=dict(
        values=[kpi["KPI"], kpi["Value"]],
        fill_color=[["#F1EFE8", "#FFFFFF"] * len(kpi)],
        font=dict(size=12),
        align="left",
        height=30
    )
)])
fig.update_layout(title="💳 Payment Summary — KPI Overview")
fig.show()

In [0]:
df = spark.sql("""
    SELECT
        payment_method,
        COUNT(*)                    AS transaction_count,
        ROUND(SUM(total_amount), 2) AS total_amount
    FROM payment_summary
    GROUP BY payment_method
    ORDER BY transaction_count DESC
""").toPandas()

fig = px.bar(
    df,
    x="payment_method",
    y="transaction_count",
    title="💳 Transaction Count by Payment Method",
    color="payment_method",
    color_discrete_sequence=px.colors.qualitative.Set2,
    text="transaction_count"
)
fig.update_traces(textposition="outside")
fig.update_layout(
    showlegend=False,
    xaxis_title="Payment Method",
    yaxis_title="Number of Transactions"
)
fig.show()

In [0]:
df = spark.sql("""
    SELECT
        payment_method,
        ROUND(SUM(total_amount), 2) AS total_amount,
        ROUND(AVG(total_amount), 2) AS avg_amount
    FROM payment_summary
    GROUP BY payment_method
    ORDER BY total_amount DESC
""").toPandas()

fig = px.bar(
    df,
    x="payment_method",
    y="total_amount",
    title="💰 Total Amount Processed by Payment Method",
    color="payment_method",
    color_discrete_sequence=px.colors.qualitative.Pastel,
    text="total_amount"
)
fig.update_traces(textposition="outside")
fig.update_layout(
    showlegend=False,
    xaxis_title="Payment Method",
    yaxis_title="Total Amount ($)"
)
fig.show()

In [0]:
df = spark.sql("""
    SELECT
        payment_status,
        COUNT(*) AS transaction_count
    FROM payment_summary
    GROUP BY payment_status
    ORDER BY transaction_count DESC
""").toPandas()

fig = px.pie(
    df,
    names="payment_status",
    values="transaction_count",
    title="📊 Payment Status Breakdown",
    hole=0.4,
    color="payment_status",
    color_discrete_map={
        "completed": "#2ECC71",
        "pending":   "#F39C12",
        "failed":    "#E74C3C",
        "refunded":  "#3498DB"
    }
)
fig.update_traces(textinfo="percent+label+value")
fig.show()

In [0]:
df = spark.sql("""
    SELECT
        payment_method,
        COUNT(CASE WHEN payment_status = 'completed' THEN 1 END) AS completed,
        COUNT(CASE WHEN payment_status = 'failed'    THEN 1 END) AS failed,
        COUNT(CASE WHEN payment_status = 'refunded'  THEN 1 END) AS refunded,
        COUNT(CASE WHEN payment_status = 'pending'   THEN 1 END) AS pending
    FROM payment_summary
    GROUP BY payment_method
    ORDER BY payment_method
""").toPandas()

fig = go.Figure()
fig.add_trace(go.Bar(name="Completed", x=df["payment_method"], y=df["completed"], marker_color="#2ECC71"))
fig.add_trace(go.Bar(name="Failed",    x=df["payment_method"], y=df["failed"],    marker_color="#E74C3C"))
fig.add_trace(go.Bar(name="Refunded",  x=df["payment_method"], y=df["refunded"],  marker_color="#3498DB"))
fig.add_trace(go.Bar(name="Pending",   x=df["payment_method"], y=df["pending"],   marker_color="#F39C12"))

fig.update_layout(
    barmode="stack",
    title="✅ Transaction Status by Payment Method",
    xaxis_title="Payment Method",
    yaxis_title="Number of Transactions",
    legend_title="Status"
)
fig.show()

In [0]:
df = spark.sql("""
    SELECT
        payment_date,
        COUNT(*)                    AS transaction_count,
        ROUND(SUM(total_amount), 2) AS daily_amount
    FROM payment_summary
    GROUP BY payment_date
    ORDER BY payment_date
""").toPandas()

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=("Daily Transaction Count", "Daily Amount Processed ($)"),
    shared_xaxes=True
)

fig.add_trace(
    go.Scatter(
        x=df["payment_date"], y=df["transaction_count"],
        mode="lines", name="Transactions",
        line=dict(color="#636EFA", width=2)
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=df["payment_date"], y=df["daily_amount"],
        mode="lines", name="Amount ($)",
        line=dict(color="#EF553B", width=2),
        fill="tozeroy"
    ),
    row=2, col=1
)

fig.update_layout(
    title_text="📅 Daily Payment Volume Over Time",
    showlegend=False,
    height=500
)
fig.show()

In [0]:
df = spark.sql("""
    SELECT
        payment_method,
        ROUND(AVG(avg_fraud_score), 4) AS avg_fraud_score,
        ROUND(MAX(avg_fraud_score), 4) AS max_fraud_score
    FROM payment_summary
    GROUP BY payment_method
    ORDER BY avg_fraud_score DESC
""").toPandas()

fig = px.bar(
    df,
    x="payment_method",
    y="avg_fraud_score",
    title="🚨 Average Fraud Score by Payment Method",
    color="avg_fraud_score",
    color_continuous_scale="Reds",
    text="avg_fraud_score"
)
fig.update_traces(textposition="outside")
fig.update_layout(
    xaxis_title="Payment Method",
    yaxis_title="Avg Fraud Score"
)
fig.show()

In [0]:
df = spark.sql("""
    SELECT
        payment_status,
        ROUND(SUM(total_amount), 2) AS total_amount,
        COUNT(*)                    AS transaction_count
    FROM payment_summary
    GROUP BY payment_status
    ORDER BY total_amount DESC
""").toPandas()

fig = px.pie(
    df,
    names="payment_status",
    values="total_amount",
    title="🌍 Total Revenue by Payment Status",
    hole=0.4,
    color="payment_status",
    color_discrete_map={
        "completed": "#2ECC71",
        "pending":   "#F39C12",
        "failed":    "#E74C3C",
        "refunded":  "#3498DB"
    }
)
fig.update_traces(textinfo="percent+label+value")
fig.show()

In [0]:
df = spark.sql("""
    SELECT
        payment_method,
        payment_status,
        ROUND(AVG(total_amount), 2) AS avg_amount
    FROM payment_summary
    GROUP BY payment_method, payment_status
""").toPandas()

pivot = df.pivot(
    index="payment_method",
    columns="payment_status",
    values="avg_amount"
).fillna(0)

fig = px.imshow(
    pivot,
    title="🔥 Avg Transaction Value — Method vs Status (Heatmap)",
    color_continuous_scale="Blues",
    text_auto=True,
    aspect="auto"
)
fig.update_layout(
    xaxis_title="Payment Status",
    yaxis_title="Payment Method"
)
fig.show()